# Notebook OSSE problem


This notebook computes the plots of the OSSE problem (Figure **01**) and of the Gihub and SEANOE repository.

In [ ]:
import xarray as xr 
import numpy as np 
import matplotlib.pyplot as plt
import cmocean
from pyinterp import fill, Axis, TemporalAxis, Grid3D, Grid2D
n_workers = 10
plt.rcdefaults()
import matplotlib.ticker as mticker 

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

import colorcet as cc

from mapping.src.functions_interp import fill_nan

In [ ]:
path_data = "../.." # Change path according to where data are saved

## Loading

### Satellite data 

In [ ]:
ds_swot = xr.open_dataset(f"{path_data}/data/OSSE/obs/dc_obs_swot/SSH_SWOT_2012-06-15.nc")

In [ ]:
date = "2012-06-15"

ds_alg_hawaii = xr.open_dataset(f"{path_data}/data/OSSE/obs/dc_obs_nadirs/alg/SSH_NADIR_{date}.nc")
ds_s3a_hawaii = xr.open_dataset(f"{path_data}/data/OSSE/obs/dc_obs_nadirs/s3a/SSH_NADIR_{date}.nc")
ds_s3b_hawaii = xr.open_dataset(f"{path_data}/data/OSSE/obs/dc_obs_nadirs/s3b/SSH_NADIR_{date}.nc")
ds_c2_hawaii = xr.open_dataset(f"{path_data}/data/OSSE/obs/dc_obs_nadirs/c2/SSH_NADIR_{date}.nc")
ds_j3_hawaii = xr.open_dataset(f"{path_data}/data/OSSE/obs/dc_obs_nadirs/j3/SSH_NADIR_{date}.nc")

In [ ]:
lon_min = 190; lon_max = 200
lat_min = 20; lat_max = 30

In [ ]:
%run ./functions_interp.ipynb

### Reference field 

In [ ]:
ds_truth = xr.open_dataset(f"{path_data}/data/OSSE/ref/2023e_SSHmapping_HF_Hawaii_eval_{date}.nc")
ds_truth = ds_truth.sel(longitude=slice(lon_min,lon_max),
                        latitude=slice(lat_min,lat_max),drop=True)
ssh_bm_truth = ds_truth.ssh_bm.load()
ssh_it_truth = ds_truth.ssh_it.load()
ssh_tot_truth = ds_truth.ssh.load()

In [ ]:
ssh_bm_truth = fill_nan(ssh_bm_truth)
ssh_it_truth = fill_nan(ssh_it_truth)
ssh_tot_truth = fill_nan(ssh_tot_truth)

### Reconstructed field 

In [ ]:
ds_QGSW = xr.open_dataset(f"{path_data}/data/mapping_outputs/config_QGSW/config_QGSW_y2012m06d15h00m00.nc")
# - BALANCED MOTIONS BM - #
ssh_bm_QGSW = ds_QGSW.ssh_bm.load()
ssh_bm_QGSW = ssh_bm_QGSW.rename({"lon":"longitude","lat":"latitude"})
# - INTERNAL TIDE IT - #
ssh_it_QGSW = ds_QGSW.ssh_it.load()
ssh_it_QGSW = ssh_it_QGSW.rename({"lon":"longitude","lat":"latitude"})

### Bathymetry field 

In [ ]:
path_bathy = "" # PATH TO BATHY, DOWNLOAD FROM GEBCO #
bathy = xr.open_dataset(path_bathy) 
bathy = bathy.assign_coords(lon=((bathy.lon % 360)))
bathy = bathy.sortby("lon")


In [ ]:
bathy_sel = bathy.sel(lon = slice(ssh_bm_truth.longitude.min(),ssh_bm_truth.longitude.max()),lat = slice(ssh_bm_truth.latitude.min(),ssh_bm_truth.latitude.max()))
bathy_sel = bathy_sel.coarsen({"lon":5,"lat":5}).mean().load()

In [ ]:

bathy_coarse = bathy.coarsen({"lon":30,"lat":30}).mean()

### Plot paper

In [ ]:
proj = ccrs.PlateCarree()

fig, ax = plt.subplots(
    2,2,
    figsize=(12,10.5),
    dpi=300,
    subplot_kw={"projection": proj}
)

v_min_bm = 100*0.7
v_max_bm = 100*1.05
v_min_it = 100*-0.06
v_max_it = 100*0.06
vmin_bathy = -7000
vmax_bathy = 0


### BATHY FIELD ###

_plt_bathy = ax[0,0].pcolormesh(
    bathy_sel.lon,
    bathy_sel.lat,
    bathy_sel.elevation,
    cmap="Blues_r",
    vmin=vmin_bathy,
    vmax=vmax_bathy,
    transform=proj
)

ax[0,0].set_title("        Bathymetry", fontsize=20)

cbar_bathy = fig.colorbar(
    _plt_bathy,
    ax=ax[0,0],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_bathy.ax.set_title("[m]", fontsize=12, pad=10)


### SAT DATA ###

_plt_swot = ax[0,1].scatter(
    ds_swot.longitude,
    ds_swot.latitude,
    c=100*(ds_swot.ssh),
    marker="o",
    edgecolors="none",
    s=1.5,
    alpha=0.8,
    cmap=cmocean.cm.haline,
    vmin=v_min_bm,
    vmax=v_max_bm,
    transform=proj,
    zorder=2
)

for _ds,_s in zip(
    [ds_alg_hawaii,ds_c2_hawaii,ds_j3_hawaii,ds_s3a_hawaii,ds_s3b_hawaii],
    1.5*np.array([1,1,1,1,1])
):

    ax[0,1].scatter(
        _ds.longitude.values,
        _ds.latitude.values,
        marker="o",
        edgecolors="none",
        s=_s,
        c=100*_ds.ssh.values,
        cmap=cmocean.cm.haline,
        vmin=v_min_bm,
        vmax=v_max_bm,
        transform=proj,
        zorder=2
    )

ax[0,1].set_facecolor('gainsboro')

cbar_swot = fig.colorbar(
    _plt_swot,
    ax=ax[0,1],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_swot.ax.set_title("[cm]", fontsize=12, pad=10)

ax[0,1].set_title("SSH observations", fontsize=20)


### BM SSH ###

_plt_bm = ax[1,0].pcolormesh(
    ds_truth.longitude,
    ds_truth.latitude,
    100*ssh_bm_truth[0,:,:],
    cmap=cmocean.cm.haline,
    vmin=v_min_bm,
    vmax=v_max_bm,
    transform=proj
)

ax[1,0].set_title("Target reference BM field", fontsize=20)

cbar_bm = fig.colorbar(
    _plt_bm,
    ax=ax[1,0],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_bm.ax.set_title("[cm]", fontsize=12, pad=10)


### IT SSH ###

_plt_it = ax[1,1].pcolormesh(
    ds_truth.longitude,
    ds_truth.latitude,
    100*ssh_it_truth[0,:,:],
    cmap='bwr',
    vmin=v_min_it,
    vmax=v_max_it,
    transform=proj
)

ax[1,1].set_title("Target reference IT field", fontsize=20)

cbar_it = fig.colorbar(
    _plt_it,
    ax=ax[1,1],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_it.ax.set_title("[cm]", fontsize=12, pad=10)


### MAP FEATURES ###

lon_ticks = np.arange(-168, -160+1, 2)
lat_ticks = np.arange(lat_min, lat_max+1, 2)

for i,_ax in enumerate(ax.flatten()):

    _ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    _ax.coastlines(resolution='50m', color='black', linewidth=1)

    _ax.add_feature(
        cfeature.LAND.with_scale('50m'),
        facecolor='darkgray',
        zorder=3
    )

    # ticks
    _ax.set_xticks(lon_ticks, crs=ccrs.PlateCarree())
    _ax.set_yticks(lat_ticks, crs=ccrs.PlateCarree())

    _ax.xaxis.set_major_formatter(LongitudeFormatter())
    _ax.yaxis.set_major_formatter(LatitudeFormatter())

    _ax.tick_params(labelsize=11)

    _ax.tick_params(
        axis='both',
        which='major',
        direction='in',
        length=5,
        width=0.8,
        labelsize=11
    )

    gl = _ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=False,
    linewidth=0.2,
    color='white',
    alpha=0.6,
    linestyle='-'
    )

    gl.xlocator = mticker.FixedLocator(lon_ticks)
    gl.ylocator = mticker.FixedLocator(lat_ticks)

fig.tight_layout()
plt.savefig("./subfig.png", transparent=False)

### Plot github and SEANOE (with reconstructions)

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

proj = ccrs.PlateCarree()

fig, ax = plt.subplots(
    2,3,
    figsize=(18,10.5),
    dpi=300,
    subplot_kw={"projection": proj}
)

v_min_bm = 100*0.7
v_max_bm = 100*1.05
v_min_it = 100*-0.06
v_max_it = 100*0.06
vmin_bathy = -7000
vmax_bathy = 0


### BATHY FIELD ###

_plt_bathy = ax[1,0].pcolormesh(
    bathy_sel.lon,
    bathy_sel.lat,
    bathy_sel.elevation,
    cmap="Blues_r",
    vmin=vmin_bathy,
    vmax=vmax_bathy,
    transform=proj
)

ax[1,0].set_title("        Bathymetry", fontsize=20)

cbar_bathy = fig.colorbar(
    _plt_bathy,
    ax=ax[1,0],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_bathy.ax.set_title("[m]", fontsize=12, pad=10)


### SAT DATA ###

_plt_swot = ax[0,0].scatter(
    ds_swot.longitude,
    ds_swot.latitude,
    c=100*(ds_swot.ssh),
    marker="o",
    edgecolors="none",
    s=1.5,
    alpha=0.8,
    cmap=cmocean.cm.haline,
    vmin=v_min_bm,
    vmax=v_max_bm,
    transform=proj,
    zorder=2
)

for _ds,_s in zip(
    [ds_alg_hawaii,ds_c2_hawaii,ds_j3_hawaii,ds_s3a_hawaii,ds_s3b_hawaii],
    1.5*np.array([1,1,1,1,1])
):

    ax[0,0].scatter(
        _ds.longitude.values,
        _ds.latitude.values,
        marker="o",
        edgecolors="none",
        s=_s,
        c=100*_ds.ssh.values,
        cmap=cmocean.cm.haline,
        vmin=v_min_bm,
        vmax=v_max_bm,
        transform=proj,
        zorder=2
    )

ax[0,0].set_facecolor('gainsboro')

cbar_swot = fig.colorbar(
    _plt_swot,
    ax=ax[0,0],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_swot.ax.set_title("[cm]", fontsize=12, pad=10)

ax[0,0].set_title("SSH observations", fontsize=20)


### REFERENCE BM SSH ###

_plt_bm = ax[0,1].pcolormesh(
    ds_truth.longitude,
    ds_truth.latitude,
    100*ssh_bm_truth[0,:,:],
    cmap=cmocean.cm.haline,
    vmin=v_min_bm,
    vmax=v_max_bm,
    transform=proj
)

ax[0,1].set_title("Target reference BM field", fontsize=20)

cbar_bm = fig.colorbar(
    _plt_bm,
    ax=ax[0,1],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_bm.ax.set_title("[cm]", fontsize=12, pad=10)


### REFERENCE IT SSH ###

_plt_it = ax[1,1].pcolormesh(
    ds_truth.longitude,
    ds_truth.latitude,
    100*ssh_it_truth[0,:,:],
    cmap='bwr',
    vmin=v_min_it,
    vmax=v_max_it,
    transform=proj
)

ax[1,1].set_title("Target reference IT field", fontsize=20)

cbar_it = fig.colorbar(
    _plt_it,
    ax=ax[1,1],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_it.ax.set_title("[cm]", fontsize=12, pad=10)

### RECONSTRUCTED BM SSH ###

_plt_bm = ax[0,2].pcolormesh(
    ssh_bm_QGSW.longitude,
    ssh_bm_QGSW.latitude,
    100*ssh_bm_QGSW[0,:,:],
    cmap=cmocean.cm.haline,
    vmin=v_min_bm,
    vmax=v_max_bm,
    transform=proj
)

ax[0,2].set_title("Reconstructed BM field", fontsize=20)

cbar_bm = fig.colorbar(
    _plt_bm,
    ax=ax[0,2],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_bm.ax.set_title("[cm]", fontsize=12, pad=10)

### RECONSTRUCTED IT SSH ###

_plt_it = ax[1,2].pcolormesh(
    ssh_it_QGSW.longitude,
    ssh_it_QGSW.latitude,
    100*ssh_it_QGSW[0,:,:],
    cmap='bwr',
    vmin=v_min_it,
    vmax=v_max_it,
    transform=proj
)

ax[1,2].set_title("Reconstructed IT field", fontsize=20)

cbar_it = fig.colorbar(
    _plt_it,
    ax=ax[1,2],
    shrink=0.92,
    extend='both',
    aspect=30
)

cbar_it.ax.set_title("[cm]", fontsize=12, pad=10)

### MAP FEATURES ###

lon_ticks = np.arange(-168, -160+1, 2)
lat_ticks = np.arange(lat_min, lat_max+1, 2)

for i,_ax in enumerate(ax.flatten()):

    _ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    _ax.coastlines(resolution='50m', color='black', linewidth=1)

    _ax.add_feature(
        cfeature.LAND.with_scale('50m'),
        facecolor='darkgray',
        zorder=3
    )

    # ticks
    _ax.set_xticks(lon_ticks, crs=ccrs.PlateCarree())
    _ax.set_yticks(lat_ticks, crs=ccrs.PlateCarree())

    _ax.xaxis.set_major_formatter(LongitudeFormatter())
    _ax.yaxis.set_major_formatter(LatitudeFormatter())

    _ax.tick_params(labelsize=11)

    _ax.tick_params(
        axis='both',
        which='major',
        direction='in',
        length=5,
        width=0.8,
        labelsize=11
    )

    gl = _ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=False,
    linewidth=0.2,
    color='white',
    alpha=0.6,
    linestyle='-'
    )

    gl.xlocator = mticker.FixedLocator(lon_ticks)
    gl.ylocator = mticker.FixedLocator(lat_ticks)

fig.tight_layout()
plt.savefig("./subfig.png", transparent=False)

### Earth Plot 

In [ ]:
lon_min = 190;lon_max =200
lat_min = 20;lat_max =30

In [ ]:
vmin_bathy=-7000;vmax_bathy=0

In [ ]:
fig = plt.figure(figsize=(5,5),dpi=300)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Orthographic(-160, 20))

# ax.add_feature(cfeature.OCEAN, color="lightgrey",zorder=0)
ax.add_feature(cfeature.LAND,color="darkgray", zorder=4, edgecolor='black')
ax.coastlines(color='black',resolution='110m', linewidth=0.3,zorder=5)

ax.plot([lon_min,lon_min,lon_max,lon_max,lon_min],[lat_min,lat_max,lat_max,lat_min,lat_min],c='red',transform=ccrs.PlateCarree(),linewidth=2)
# ax.scatter([lon_min,lon_min,lon_max,lon_max],[lat_min,lat_max,lat_min,lat_max],c='red',transform=ccrs.PlateCarree(),s=20)
ax.pcolormesh(bathy_coarse.lon,bathy_coarse.lat,bathy_coarse.elevation,cmap="Blues_r",transform=ccrs.PlateCarree(),vmin=vmin_bathy,vmax=vmax_bathy)

ax.set_global()
# ax.gridlines()

plt.savefig("./earth.png", transparent=True)